In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, auc, f1_score, hamming_loss
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from scipy.signal import stft

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
MODELS_DIR, RESULTS_DIR = output_dirs()
X_train, y_train, X_test, y_test, class_names, X_feat_train, X_feat_test = load_cache()
tf.keras.utils.set_random_seed(SEED)  # reproducible weights init / shuffling

Lead set  -> lead1 (channels [0])
Signals   -> Train: (19230, 1000, 1) | Test: (2158, 1000, 1)
Features  -> Train: (19230, 24) | Test: (2158, 24)
Targets   -> Classes (5): ['CD', 'HYP', 'MI', 'NORM', 'STTC']


In [3]:

def resnet_block_1d(x, filters, stride=1):
    shortcut = x
    y = layers.Conv1D(filters, 7, strides=stride, padding="same")(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    y = layers.Conv1D(filters, 7, strides=1, padding="same")(y)
    y = layers.BatchNormalization()(y)
    
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, strides=stride, padding="same")(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    out = layers.add([shortcut, y])
    return layers.ReLU()(out)

def build_sota_resnet_attention(input_shape=(1000, 3), num_classes=5):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv1D(64, 7, strides=2, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    x = resnet_block_1d(x, 64, stride=1)
    x = resnet_block_1d(x, 64, stride=1)
    x = resnet_block_1d(x, 128, stride=2)
    x = resnet_block_1d(x, 128, stride=1)
    x = resnet_block_1d(x, 256, stride=2)
    x = resnet_block_1d(x, 256, stride=1)
    
    # Multi-Head Attention mechanism over temporal features
    attn_out = layers.MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
    x = layers.LayerNormalization()(layers.add([x, attn_out]))
    
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="sigmoid")(x)
    
    return models.Model(inputs=inputs, outputs=outputs, name="SOTA_ResNet_Attention")

sota_model = build_sota_resnet_attention(input_shape=(X_train.shape[1], X_train.shape[2]), num_classes=len(class_names))
sota_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(multi_label=True, name="macro_auc")]
)

print("\nTraining Deep ResNet + Attention Ceiling Benchmark...")
cb = [
    callbacks.EarlyStopping(monitor="val_macro_auc", mode="max", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)
]
sota_model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=25,
    batch_size=64,
    callbacks=cb,
    verbose=1
)
sota_probs = sota_model.predict(X_test, batch_size=128)
print(f"Deep ResNet + Attention Macro AUC: {roc_auc_score(y_test, sota_probs, average='macro'):.4f}")


Training Deep ResNet + Attention Ceiling Benchmark...
Epoch 1/25


  1/271 ━━━━━━━━━━━━━━━━━━━━ 41:22 9s/step - loss: 0.8774 - macro_auc: 0.4723

  2/271 ━━━━━━━━━━━━━━━━━━━━ 1:43 383ms/step - loss: 0.8035 - macro_auc: 0.5263

  3/271 ━━━━━━━━━━━━━━━━━━━━ 1:39 371ms/step - loss: 0.7307 - macro_auc: 0.5455

  4/271 ━━━━━━━━━━━━━━━━━━━━ 1:35 360ms/step - loss: 0.6835 - macro_auc: 0.5597

  5/271 ━━━━━━━━━━━━━━━━━━━━ 1:34 353ms/step - loss: 0.6724 - macro_auc: 0.5530

... (2,739 outputs left out)

16/17 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step

17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 221ms/step


Deep ResNet + Attention Macro AUC: 0.8310


In [4]:
# For SOTA ResNet (Notebook 5)
np.savez_compressed(RESULTS_DIR / "preds_sota.npz", sota=sota_probs)
sota_model.save(MODELS_DIR / "sota_resnet.keras")
print("Saved predictions and model weights!")

Saved predictions and model weights!
